In [1]:
# Remove any existing torch-related packages to avoid conflicts
!pip uninstall torch torchvision torchaudio -y
!rm -rf /usr/local/lib/python3.11/dist-packages/torch*  # Remove corrupted installations
!rm -rf /usr/local/lib/python3.11/dist-packages/~orch*  # Remove invalid distributions

# Install dependencies with exact versions
!pip install --no-cache-dir torch==2.3.0 torchvision==0.18.0 -f https://download.pytorch.org/whl/cu121 -U -q
!pip install transformers==4.45.2 datasets==2.21.0 peft==0.12.0 sacrebleu==2.4.3 nltk==3.8.1 rouge-score==0.1.2 pycocoevalcap==1.2 -q
!pip install accelerate==0.34.2 bitsandbytes==0.44.0 -U -q
!apt-get update -qq && apt-get install -y default-jre -qq  # Required for SPICE
!pip install pillow==10.4.0 -q  # Ensure compatible PIL version

# Verify installed versions
import importlib.metadata
print(f"Installed torch version: {importlib.metadata.version('torch')}")
print(f"Installed torchvision version: {importlib.metadata.version('torchvision')}")

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # For debugging CUDA errors
print("Dependencies installed. **Please restart the runtime (Runtime > Restart runtime) and then run Cell 2.**")

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.2/779.2 MB 203.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 227.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 273.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 211.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 233.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 331.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 214.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# Set environment variable to avoid circular import issues
import os
os.environ['TORCHVISION_DISABLE_EXTENSION'] = '1'  # Disable torchvision extensions

# Check dependency versions
import sys
print(f"Python version: {sys.version}")
import importlib.metadata
print(f"Installed torch version (metadata): {importlib.metadata.version('torch')}")
print(f"Installed torchvision version (metadata): {importlib.metadata.version('torchvision')}")
import torch
print(f"PyTorch version (module): {torch.__version__}")
import torchvision
print(f"Torchvision version (module): {torchvision.__version__}")
import transformers
print(f"Transformers version: {transformers.__version__}")

# Import libraries
from PIL import Image
from datasets import Dataset, DatasetDict
from transformers import Blip2Processor, Blip2ForConditionalGeneration, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import nltk
from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from pycocoevalcap.spice.spice import Spice
from pycocoevalcap.cider.cider import Cider
import random
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files, drive
import numpy as np

# Download NLTK data
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("Libraries imported and NLTK data downloaded.")

Python version: 3.11.12 (main, Apr  9 2025, 08:55:54) [GCC 11.4.0]
Installed torch version (metadata): 2.3.0
Installed torchvision version (metadata): 0.18.0
PyTorch version (module): 2.3.0+cu121
Torchvision version (module): 0.18.0+cu121
Transformers version: 4.45.2
Libraries imported and NLTK data downloaded.


In [4]:
# Setup environment and download dataset
def setup_environment():
    try:
        print("Uploading kaggle.json...")
        uploaded = files.upload()  # Upload kaggle.json
        os.makedirs('/root/.kaggle', exist_ok=True)
        os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
        os.chmod('/root/.kaggle/kaggle.json', 600)
        print("Downloading Flickr8k dataset...")
        !kaggle datasets download -d adityajn105/flickr8k --force
        !unzip -o -q flickr8k.zip -d flickr8k
        print("Dataset downloaded and extracted.")
    except Exception as e:
        print(f"Error setting up environment: {e}")
        raise

setup_environment()

Uploading kaggle.json...


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/adityajn105/flickr8k
License(s): CC0-1.0
 91% 965M/1.04G [00:11<00:02, 44.5MB/s]
100% 1.04G/1.04G [00:11<00:00, 99.1MB/s]
Dataset downloaded and extracted.


In [5]:
# Load and preprocess Flickr8k dataset
def load_flickr8k_data():
    try:
        print("Loading Flickr8k dataset...")
        with open('flickr8k/captions.txt', 'r') as f:
            lines = f.readlines()[1:]  # Skip header
        image_paths = []
        captions = []
        image_to_captions = {}
        for line in lines:
            img, cap = line.strip().split(',', 1)
            img_path = os.path.join('flickr8k/Images', img)
            if os.path.exists(img_path):
                image_paths.append(img_path)
                captions.append(cap)
                if img_path not in image_to_captions:
                    image_to_captions[img_path] = []
                image_to_captions[img_path].append(cap)

        # Subsample to ~5,000 pairs (1 caption per image)
        unique_images = list(set(image_paths))
        selected_images = random.sample(unique_images, min(5000, len(unique_images)))
        image_paths = []
        captions = []
        for img in selected_images:
            image_paths.append(img)
            captions.append(random.choice(image_to_captions[img]))

        # Create dataset
        data = {'image_path': image_paths, 'caption': captions}
        dataset = Dataset.from_dict(data)

        # Split: 80% train, 10% val, 10% test
        train_test = dataset.train_test_split(test_size=0.2, seed=42)
        val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)
        datasets = DatasetDict({
            'train': train_test['train'],
            'validation': val_test['train'],
            'test': val_test['test']
        })
        print(f"Dataset splits: {len(datasets['train'])} train, {len(datasets['validation'])} val, {len(datasets['test'])} test")

        return datasets, image_to_captions
    except Exception as e:
        print(f"Error loading dataset: {e}")
        raise

datasets, all_captions = load_flickr8k_data()

Loading Flickr8k dataset...
Dataset splits: 4000 train, 500 val, 500 test


In [9]:
# Preprocess data for model input
def preprocess_data(example, processor):
    try:
        image = Image.open(example['image_path']).convert('RGB')
        inputs = processor(
            images=image,
            text=example['caption'],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=128
        )
        return {
            'pixel_values': inputs.pixel_values.squeeze(0),
            'input_ids': inputs.input_ids.squeeze(0),
            'attention_mask': inputs.attention_mask.squeeze(0),
            'labels': inputs.input_ids.squeeze(0)  # Labels are same as input_ids for generation
        }
    except Exception as e:
        print(f"Error preprocessing example {example['image_path']}: {e}")
        return None

print("Preprocessing function defined.")

Preprocessing function defined.


In [10]:
# Setup model and processor
def setup_model_and_processor(fine_tuning_strategy="lora"):
    try:
        print(f"Setting up model with {fine_tuning_strategy} strategy...")
        processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

        quantization_config = BitsAndBytesConfig(
            load_in_8bit=True,
            bnb_8bit_compute_dtype=torch.float16,
            bnb_8bit_use_double_quant=True
        ) if fine_tuning_strategy != "full" else None

        model = Blip2ForConditionalGeneration.from_pretrained(
            "Salesforce/blip2-opt-2.7b",
            torch_dtype=torch.float16,
            device_map="auto",
            quantization_config=quantization_config
        )

        if fine_tuning_strategy == "lora":
            lora_config = LoraConfig(
                r=16,
                lora_alpha=32,
                target_modules=["q_proj", "v_proj"],
                lora_dropout=0.05,
                bias="none",
                task_type="CAUSAL_LM"
            )
            model = get_peft_model(model, lora_config)
            model.print_trainable_parameters()
        elif fine_tuning_strategy == "layer_freeze":
            for param in model.vision_model.parameters():
                param.requires_grad = False

        return model, processor
    except Exception as e:
        print(f"Error setting up model: {e}")
        raise

model, processor = setup_model_and_processor(fine_tuning_strategy="lora")

Setting up model with lora strategy...


Unused kwargs: ['bnb_8bit_compute_dtype', 'bnb_8bit_use_double_quant']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 5,242,880 || all params: 3,750,004,736 || trainable%: 0.1398


In [1]:
# Train the model
def train_model(model, processor, datasets, fine_tuning_strategy="lora"):
    try:
        print(f"Training model with {fine_tuning_strategy}...")
        # Preprocess datasets
        processed_datasets = datasets.map(
            lambda x: preprocess_data(x, processor),
            remove_columns=['image_path', 'caption'],
            num_proc=1  # Avoid multiprocessing issues in Colab
        ).filter(lambda x: x is not None)  # Remove failed examples

        # Custom data collator to ensure correct batching
        from torch.nn.utils.rnn import pad_sequence
        def custom_data_collator(features):
            batch = {}
            for key in ['pixel_values', 'input_ids', 'attention_mask', 'labels']:
                if key in features[0]:
                    if key == 'pixel_values':
                        batch[key] = torch.stack([f[key] for f in features])
                    else:
                        batch[key] = pad_sequence([f[key] for f in features], batch_first=True, padding_value=processor.tokenizer.pad_token_id if key != 'labels' else -100)
            return batch

        training_args = TrainingArguments(
            output_dir=f"./blip2_finetuned_{fine_tuning_strategy}",
            num_train_epochs=3,
            per_device_train_batch_size=4,
            per_device_eval_batch_size=4,
            gradient_accumulation_steps=2,
            eval_strategy="epoch",
            save_strategy="epoch",
            logging_steps=50,
            learning_rate=5e-5,
            load_best_model_at_end=True,
            fp16=True,
            remove_unused_columns=True,  # Changed to True
            dataloader_num_workers=0,
            report_to="none",
            run_name=f"blip2_lora_finetune_{fine_tuning_strategy}"
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=processed_datasets['train'],
            eval_dataset=processed_datasets['validation'],
            data_collator=custom_data_collator
        )

        trainer.train()
        return model
    except Exception as e:
        print(f"Error training model: {e}")
        raise

model = train_model(model, processor, datasets, fine_tuning_strategy="lora")

NameError: name 'model' is not defined

In [ ]:
# Generate captions
def generate_caption(model, processor, image_path, method="beam", **kwargs):
    try:
        image = Image.open(image_path).convert('RGB')
        inputs = processor(images=image, return_tensors="pt").to("cuda", torch.float16)

        model.eval()
        with torch.no_grad():
            if method == "beam":
                outputs = model.generate(
                    **inputs,
                    max_length=50,
                    num_beams=kwargs.get("num_beams", 5),
                    length_penalty=1.0,
                    num_return_sequences=1
                )
            elif method == "top_p":
                outputs = model.generate(
                    **inputs,
                    max_length=50,
                    do_sample=True,
                    top_p=kwargs.get("top_p", 0.9),
                    temperature=kwargs.get("temperature", 0.7),
                    num_return_sequences=1
                )
            else:
                raise ValueError(f"Unsupported decoding method: {method}")

        caption = processor.decode(outputs[0], skip_special_tokens=True)
        return caption
    except Exception as e:
        print(f"Error generating caption for {image_path}: {e}")
        return ""

print("Caption generation function defined.")

In [ ]:
# Evaluate model (quantitative metrics)
def evaluate_model(model, processor, test_dataset, all_captions):
    try:
        print("Evaluating model...")
        references = []
        hypotheses_beam = []
        hypotheses_top_p = []

        for example in test_dataset:
            image_path = example['image_path']
            ref_captions = all_captions.get(image_path, [example['caption']])
            hyp_caption_beam = generate_caption(model, processor, image_path, method="beam", num_beams=5)
            hyp_caption_top_p = generate_caption(model, processor, image_path, method="top_p", top_p=0.9, temperature=0.7)

            references.append(ref_captions)
            hypotheses_beam.append(hyp_caption_beam)
            hypotheses_top_p.append(hyp_caption_top_p)

        metrics_beam = compute_metrics(references, hypotheses_beam, "beam")
        metrics_top_p = compute_metrics(references, hypotheses_top_p, "top_p")

        results = {"beam": metrics_beam, "top_p": metrics_top_p}
        df = pd.DataFrame({
            "Metric": metrics_beam.keys(),
            "Beam Search": metrics_beam.values(),
            "Top-p Sampling": metrics_top_p.values()
        })
        df.to_csv("evaluation_metrics.csv", index=False)
        return results
    except Exception as e:
        print(f"Error evaluating model: {e}")
        raise

def compute_metrics(references, hypotheses, method_name):
    try:
        bleu4 = corpus_bleu([[ref] for ref in references for _ in ref], hypotheses, weights=(0.25, 0.25, 0.25, 0.25))

        meteor_scores = [meteor_score(ref, hyp) for ref, hyp in zip(references, hypotheses)]
        meteor_avg = np.mean(meteor_scores)

        rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
        rouge_scores = [rouge.score(ref[0], hyp)['rougeL'].fmeasure for ref, hyp in zip(references, hypotheses)]
        rouge_avg = np.mean(rouge_scores)

        gts = {i: ref for i, ref in enumerate(references)}
        res = {i: [hyp] for i, hyp in enumerate(hypotheses)}
        spice_scorer = Spice()
        cider_scorer = Cider()
        spice_score, _ = spice_scorer.compute_score(gts, res)
        cider_score, _ = cider_scorer.compute_score(gts, res)

        self_bleu = corpus_bleu([[h] for h in hypotheses[:-1]], hypotheses[1:], weights=(0.25, 0.25, 0.25, 0.25))

        distinct_2 = len(set([tuple(h.split()[:2]) for h in hypotheses if len(h.split()) >= 2])) / len(hypotheses)

        return {
            "BLEU-4": bleu4 * 100,
            "METEOR": meteor_avg * 100,
            "ROUGE-L": rouge_avg * 100,
            "SPICE": spice_score * 100,
            "CIDEr": cider_score * 100,
            "Self-BLEU": self_bleu * 100,
            "Distinct-2": distinct_2 * 100
        }
    except Exception as e:
        print(f"Error computing metrics for {method_name}: {e}")
        return {k: 0.0 for k in ["BLEU-4", "METEOR", "ROUGE-L", "SPICE", "CIDEr", "Self-BLEU", "Distinct-2"]}

metrics = evaluate_model(model, processor, datasets['test'], datasets['all_captions'])
print("Evaluation Metrics:")
for method, scores in metrics.items():
    print(f"\n{method.upper()}:")
    for k, v in scores.items():
        print(f"{k}: {v:.2f}")

In [ ]:
# Qualitative and error analysis
def qualitative_analysis(model, processor, test_dataset, all_captions, num_examples=20):
    try:
        print("Performing qualitative analysis...")
        examples = random.sample(list(test_dataset), min(num_examples, len(test_dataset)))
        analysis = []

        for example in examples:
            image_path = example['image_path']
            ref_captions = all_captions.get(image_path, [example['caption']])
            hyp_caption_beam = generate_caption(model, processor, image_path, method="beam", num_beams=5)
            hyp_caption_top_p = generate_caption(model, processor, image_path, method="top_p", top_p=0.9, temperature=0.7)

            has_hallucination = any(word in hyp_caption_beam.lower() for word in ["imaginary", "not present"]) or \
                               len(set(hyp_caption_beam.lower().split()) - set(' '.join(ref_captions).lower().split())) > 5
            has_repetition = len(set(hyp_caption_beam.split())) < len(hyp_caption_beam.split()) * 0.8
            key_elements = sum(1 for word in hyp_caption_beam.lower().split() if word in ' '.join(ref_captions).lower()) >= 3

            analysis.append({
                "image_path": image_path,
                "reference": ref_captions[0],
                "all_references": ref_captions,
                "beam_caption": hyp_caption_beam,
                "top_p_caption": hyp_caption_top_p,
                "hallucination": has_hallucination,
                "repetition": has_repetition,
                "has_key_elements": key_elements
            })

        df = pd.DataFrame(analysis)
        df.to_csv("qualitative_analysis.csv", index=False)

        for i in range(min(3, len(analysis))):
            img = Image.open(analysis[i]['image_path'])
            plt.figure(figsize=(8, 6))
            plt.imshow(img)
            plt.title(f"Ref: {analysis[i]['reference']}\nBeam: {analysis[i]['beam_caption']}\nTop-p: {analysis[i]['top_p_caption']}")
            plt.axis('off')
            plt.show()

        print("Iterating on decoding parameters...")
        test_image = test_dataset[0]['image_path']
        for temp in [0.5, 0.7, 1.0]:
            caption = generate_caption(model, processor, test_image, method="top_p", top_p=0.9, temperature=temp)
            print(f"Temperature {temp}: {caption}")

        return analysis
    except Exception as e:
        print(f"Error in qualitative analysis: {e}")
        raise

analysis = qualitative_analysis(model, processor, datasets['test'], datasets['all_captions'])

In [ ]:
# Save outputs and export to Google Drive
def save_outputs(model, processor):
    try:
        print("Saving outputs...")
        model.save_pretrained("./blip2_finetuned_final")
        processor.save_pretrained("./blip2_finetuned_final")

        drive.mount('/content/drive')
        !cp -r blip2_finetuned_final /content/drive/MyDrive/
        !cp evaluation_metrics.csv /content/drive/MyDrive/
        !cp qualitative_analysis.csv /content/drive/MyDrive/
        print("Outputs saved to Google Drive.")
    except Exception as e:
        print(f"Error saving outputs: {e}")
        raise

save_outputs(model, processor)